# 🎂 סרטון מזל טוב לאבא — 4 דקות
117 תמונות + סרטונים + שירים, עד 4 דקות סה"כ.

**הרץ את התאים לפי הסדר.**

In [ ]:
# תא 1 — התקנת כלים + פונט עברית + חיבור Drive
print('מתקין כלים...')
!apt-get install -y ffmpeg fonts-noto fonts-noto-color-emoji libheif-dev libheif-examples > /dev/null 2>&1
!pip install Pillow pillow-heif numpy -q

import pillow_heif
pillow_heif.register_heif_opener()  # פתיחת HEIC/HEIF

from google.colab import drive
drive.mount('/content/drive')

import subprocess
font_check = subprocess.run(
    "find /usr/share/fonts -name '*Hebrew*.ttf' -o -name 'NotoSans-*.ttf' | head -3",
    shell=True, capture_output=True, text=True
).stdout.strip()
print(f'\nפונטים עבריים:\n{font_check}\n')
print('✓ מוכן')

In [ ]:
# תא 2 — הגדרת נתיבים
import os, glob, shutil

DRIVE_PHOTOS  = '/content/drive/MyDrive/עדכון תמונות לאבא'
DRIVE_MUSIC   = '/content/drive/MyDrive/Claude/יומולדת של אבא'

WORK_DIR      = '/content/mazel_tov'
PHOTOS_DIR    = f'{WORK_DIR}/photos_raw'
PHOTOS_FIXED  = f'{WORK_DIR}/photos_fixed'
VIDEOS_DIR    = f'{WORK_DIR}/videos'
MUSIC_DIR     = f'{WORK_DIR}/music'
OUTPUT_DIR    = f'{WORK_DIR}/output'
SEGMENTS_DIR  = f'{WORK_DIR}/segments'

for d in [PHOTOS_DIR, PHOTOS_FIXED, VIDEOS_DIR, MUSIC_DIR, OUTPUT_DIR, SEGMENTS_DIR]:
    os.makedirs(d, exist_ok=True)

ok1 = os.path.exists(DRIVE_PHOTOS)
ok2 = os.path.exists(DRIVE_MUSIC)
print(f'תיקיית תמונות: {"✓" if ok1 else "❌"} {DRIVE_PHOTOS}')
print(f'תיקיית מוזיקה: {"✓" if ok2 else "❌"} {DRIVE_MUSIC}')
if ok1:
    all_files = os.listdir(DRIVE_PHOTOS)
    print(f'\nסה"כ קבצים בתיקייה: {len(all_files)}')

In [ ]:
# תא 3 — העתקה חכמה: כל פורמט תמונה + כל וידאו
from PIL import Image
import pillow_heif
pillow_heif.register_heif_opener()

VIDEO_EXTS = {'.mp4','.mov','.avi','.mkv','.webm','.m4v'}

def is_image(path):
    """בדוק אם הקובץ הוא תמונה — דרך PIL, לא דרך סיומת בלבד."""
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

photos_copied = videos_copied = unknown = 0
all_files = sorted(os.listdir(DRIVE_PHOTOS))
print(f'בודק {len(all_files)} קבצים...\n')

for fname in all_files:
    src = os.path.join(DRIVE_PHOTOS, fname)
    if not os.path.isfile(src):
        continue
    ext = os.path.splitext(fname)[1].lower()

    if ext in VIDEO_EXTS:
        dst = os.path.join(VIDEOS_DIR, fname)
        if not os.path.exists(dst): shutil.copy2(src, dst)
        videos_copied += 1
    elif is_image(src):  # פתיחה אמיתית עם PIL — תופס כל פורמט
        dst = os.path.join(PHOTOS_DIR, fname)
        if not os.path.exists(dst): shutil.copy2(src, dst)
        photos_copied += 1
    else:
        print(f'  ⚠️  קובץ לא מזוהה: {fname}')
        unknown += 1

# מוזיקה — מהתיקייה הקודמת
music_copied = 0
for fname in os.listdir(DRIVE_MUSIC):
    src = os.path.join(DRIVE_MUSIC, fname)
    if not os.path.isfile(src): continue
    ext = os.path.splitext(fname)[1].lower()
    size = os.path.getsize(src)
    # קובצי וידאו קטנים = קליפים מוזיקליים
    if ext in VIDEO_EXTS and size < 5_000_000:
        # שלומי שבת / אביתר בנאי בלבד
        if any(k in fname for k in ['שלומי','אביתר','shlomi','avitar','Shlomi','Avitar']):
            dst = os.path.join(MUSIC_DIR, fname)
            if not os.path.exists(dst): shutil.copy2(src, dst)
            music_copied += 1

print(f'\n📸 תמונות: {photos_copied}')
print(f'🎬 סרטונים: {videos_copied}')
print(f'🎵 שירים: {music_copied}  →  {[os.path.basename(f) for f in os.listdir(MUSIC_DIR)]}')
print(f'❓ לא מזוהה: {unknown}')
print(f'סה"כ: {photos_copied + videos_copied + unknown} / {len(all_files)}')

In [ ]:
# תא 4 — תיקון סיבוב + שדרוג ל-HD + חידוד מטושטשות
from PIL import Image, ImageOps, ImageFilter
import numpy as np

TARGET_W = 1920  # רוחב יעד HD

def blur_score(img):
    g = np.array(img.convert('L'), dtype=float)
    return float(np.var(g[1:]-g[:-1]) + np.var(g[:,1:]-g[:,:-1]))

photos_raw = sorted(glob.glob(f'{PHOTOS_DIR}/*'))
print(f'מעבד {len(photos_raw)} תמונות...\n')

fixed_paths = []
upscaled = sharpened = errors = 0

for i, src in enumerate(photos_raw):
    dst = os.path.join(PHOTOS_FIXED, f'{i:04d}.jpg')
    if os.path.exists(dst):
        fixed_paths.append(dst)
        continue

    try:
        img = Image.open(src)
        img = ImageOps.exif_transpose(img)   # סיבוב נכון
        img = img.convert('RGB')

        # שדרוג לתמונות ברזולוציה נמוכה — מעלה ל-HD
        w, h = img.size
        if w < TARGET_W:
            scale = TARGET_W / w
            img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
            upscaled += 1

        # חידוד תמיד קל + חידוד חזק לתמונות מטושטשות
        score = blur_score(img)
        if score < 800:  # מטושטש
            img = img.filter(ImageFilter.UnsharpMask(radius=2, percent=200, threshold=2))
            sharpened += 1
        else:
            img = img.filter(ImageFilter.UnsharpMask(radius=1, percent=80, threshold=3))

        img.save(dst, 'JPEG', quality=92)
        fixed_paths.append(dst)
    except Exception as e:
        print(f'  ❌ {os.path.basename(src)}: {e}')
        errors += 1

    if (i+1) % 25 == 0:
        print(f'  עבדו {i+1}/{len(photos_raw)}')

print(f'\n✓ תמונות מוכנות: {len(fixed_paths)}/{len(photos_raw)}')
print(f'   שודרגו ל-HD: {upscaled}')
print(f'   חודדו (מטושטשות): {sharpened}')
print(f'   שגיאות: {errors}')

In [ ]:
# תא 5 — בדיקת מוזיקה
music_files = sorted(glob.glob(f'{MUSIC_DIR}/*'))
print('🎵 שירים שנמצאו:')
for i, f in enumerate(music_files):
    size = os.path.getsize(f) / 1024
    print(f'  [{i}] {os.path.basename(f)}  ({size:.0f}KB)')

if not music_files:
    raise Exception('❌ לא נמצאה מוזיקה! בדוק את DRIVE_MUSIC')

# סדר: שלומי שבת קודם, אביתר בנאי שני
shlomi = [f for f in music_files if 'שלומי' in f or 'shlomi' in f.lower()]
avitar = [f for f in music_files if 'אביתר' in f or 'avitar' in f.lower()]

MUSIC1 = shlomi[0] if shlomi else music_files[0]
MUSIC2 = avitar[0] if avitar else (music_files[1] if len(music_files)>1 else music_files[0])

print(f'\n▶️  שיר 1: {os.path.basename(MUSIC1)}')
print(f'▶️  שיר 2: {os.path.basename(MUSIC2)}')

In [ ]:
# תא 6 — פונקציות הבנייה (מותאם לסרטון של 4 דקות)
import subprocess

W, H = 1920, 1080
FPS  = 25

# *** משכי זמן לסרטון של 4 דקות ***
PHOTO_DURATION = 1.8   # שניות לכל תמונה
VIDEO_MAX_DUR  = 2.5   # שניות מקסימום לכל קליפ
TITLE_DUR      = 5     # שקף פתיחה
END_DUR        = 7     # שקף סיום

TITLE_TEXT = 'אבא היקר\nהמון מזל טוב'
END_TEXT   = 'המון מזל טוב\nאוהבים אותך\nמשפחת פיינגרש'

def run(cmd, desc=''):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ❌ {desc}: {r.stderr[-250:]}')
        return False
    return True

def find_hebrew_font():
    for f in [
        '/usr/share/fonts/truetype/noto/NotoSansHebrew-Regular.ttf',
        '/usr/share/fonts/truetype/noto/NotoSansHebrew-Bold.ttf',
        '/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    ]:
        if os.path.exists(f): return f
    return subprocess.run("find /usr/share/fonts -name '*.ttf' | head -1",
                          shell=True, capture_output=True, text=True).stdout.strip()

def get_duration(path):
    r = subprocess.run(
        f'ffprobe -v quiet -show_entries format=duration -of csv=p=0 "{path}"',
        shell=True, capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return PHOTO_DURATION

def make_title_slide(text, filename, duration, bg='0a0a1a'):
    out = f'{SEGMENTS_DIR}/{filename}'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    font = find_hebrew_font()
    lines = text.split('\n')
    line_h = 115
    start_y = (H - len(lines)*line_h) // 2
    parts = []
    for i, line in enumerate(lines):
        esc = line.replace("'", "\\'").replace(':', '\\:')
        size = 100 if i == 0 else 80
        color = 'white' if i == 0 else '#FFD700'
        y = start_y + i*line_h
        parts.append(
            f"drawtext=fontfile='{font}':text='{esc}':fontcolor={color}:"
            f"fontsize={size}:x=(w-text_w)/2:y={y}:"
            f"shadowcolor=black:shadowx=3:shadowy=3"
        )
    vf = ','.join(parts) + f',fade=t=in:st=0:d=1,fade=t=out:st={duration-1}:d=1'
    cmd = (f'ffmpeg -y -f lavfi -i color=c={bg}:{W}x{H}:rate={FPS}:duration={duration} '
           f'-vf "{vf}" -c:v libx264 -preset fast -crf 18 -pix_fmt yuv420p "{out}"')
    print(f'  שקופית: {filename}')
    run(cmd, filename)
    return out

def make_photo_segment(photo_path, idx, duration=PHOTO_DURATION):
    out = f'{SEGMENTS_DIR}/photo_{idx:03d}.mp4'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    BIG_W, BIG_H = 2112, 1188
    total_frames = max(2, int(duration * FPS))
    scale = (
        f'scale={BIG_W}:{BIG_H}:force_original_aspect_ratio=decrease,'
        f'pad={BIG_W}:{BIG_H}:(ow-iw)/2:(oh-ih)/2:black,'
        f'scale=trunc(iw/2)*2:trunc(ih/2)*2'
    )
    # Ken Burns קל (מתאים למשך קצר)
    if   idx % 3 == 0: z = "'min(zoom+0.0008,1.05)'"
    elif idx % 3 == 1: z = "'if(lte(zoom,1.0),1.04,max(1.0,zoom-0.0008))'"
    else:              z = "'1.02'"
    zoom = (f"zoompan=z={z}:d={total_frames}:"
            f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={W}x{H}:fps={FPS}")
    fo = max(0, duration - 0.25)
    fade = f'fade=t=in:st=0:d=0.25,fade=t=out:st={fo}:d=0.25'
    cmd = (
        f'ffmpeg -y -loop 1 -i "{photo_path}" '
        f'-vf "{scale},{zoom},{fade}" '
        f'-t {duration} -r {FPS} -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p "{out}"'
    )
    ok = run(cmd, f'photo {idx+1}')
    if not ok and os.path.exists(out) and os.path.getsize(out) == 0:
        os.remove(out)
    return out

def make_video_segment(video_path, idx, max_dur=VIDEO_MAX_DUR):
    out = f'{SEGMENTS_DIR}/video_{idx:03d}.mp4'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    dur = get_duration(video_path)
    trim = min(dur, max_dur)
    start = (dur - max_dur) / 3 if dur > max_dur else 0
    fo = max(0, trim - 0.25)
    vf = (
        f'scale={W}:{H}:force_original_aspect_ratio=decrease,'
        f'pad={W}:{H}:(ow-iw)/2:(oh-ih)/2:black,'
        f'fade=t=in:st=0:d=0.25,fade=t=out:st={fo}:d=0.25'
    )
    cmd = (
        f'ffmpeg -y -ss {start:.2f} -i "{video_path}" -t {trim:.2f} '
        f'-vf "{vf}" -r {FPS} -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p -an "{out}"'
    )
    run(cmd, f'video {idx+1}')
    return out

def mix_music(total_dur):
    out = f'{WORK_DIR}/mixed_audio.aac'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    d1 = get_duration(MUSIC1)
    d2 = get_duration(MUSIC2)
    p1 = min(total_dur * 0.55, d1)
    p2 = min(total_dur - p1, d2)
    fade = 2.0
    cmd = (
        f'ffmpeg -y -i "{MUSIC1}" -i "{MUSIC2}" -filter_complex '
        f'"[0:a]atrim=0:{p1},asetpts=PTS-STARTPTS,'
        f'afade=t=in:st=0:d={fade},afade=t=out:st={p1-fade}:d={fade}[a1];'
        f'[1:a]atrim=0:{p2},asetpts=PTS-STARTPTS,'
        f'afade=t=in:st=0:d={fade},afade=t=out:st={p2-fade}:d={fade}[a2];'
        f'[a1][a2]concat=n=2:v=0:a=1[aout]" '
        f'-map [aout] -c:a aac -b:a 192k "{out}"'
    )
    print(f'  מיקס: {p1:.0f}ש שלומי + {p2:.0f}ש אביתר')
    run(cmd, 'music')
    return out

print(f'✓ פונקציות מוגדרות')
print(f'   פונט: {find_hebrew_font()}')
print(f'   תמונה: {PHOTO_DURATION}ש | וידאו מקס: {VIDEO_MAX_DUR}ש')

In [ ]:
# תא 7 — בניית הסגמנטים
photos = fixed_paths
videos = sorted(glob.glob(f'{VIDEOS_DIR}/*'))
videos = [v for v in videos if os.path.splitext(v)[1].lower() in VIDEO_EXTS]

print(f'📸 תמונות: {len(photos)}')
print(f'🎬 סרטונים: {len(videos)}')

expected_dur = (
    TITLE_DUR + END_DUR +
    len(photos) * PHOTO_DURATION +
    len(videos) * VIDEO_MAX_DUR
)
print(f'\nאורך צפוי: {expected_dur:.0f}ש ({expected_dur/60:.1f} דקות)')

segments = []

# שקף פתיחה
print('\n--- שקף פתיחה ---')
seg = make_title_slide(TITLE_TEXT, 'title.mp4', TITLE_DUR, bg='0a0a1a')
if os.path.exists(seg) and os.path.getsize(seg) > 0:
    segments.append(seg)

# תמונות + סרטונים — קטע וידאו אחרי כל 12 תמונות
print('\n--- תמונות + סרטונים ---')
vi = 0
video_every = max(1, len(photos) // max(1, len(videos)+1)) if videos else 999
for i, photo in enumerate(photos):
    seg = make_photo_segment(photo, i)
    if os.path.exists(seg) and os.path.getsize(seg) > 0:
        segments.append(seg)
    if videos and (i+1) % video_every == 0 and vi < len(videos):
        seg = make_video_segment(videos[vi], vi)
        if os.path.exists(seg) and os.path.getsize(seg) > 0:
            segments.append(seg)
        vi += 1
    if (i+1) % 20 == 0:
        print(f'  ...{i+1}/{len(photos)}')

# וידאו שנותרו
while vi < len(videos):
    seg = make_video_segment(videos[vi], vi)
    if os.path.exists(seg) and os.path.getsize(seg) > 0:
        segments.append(seg)
    vi += 1

# שקף סיום
print('\n--- שקף סיום ---')
seg = make_title_slide(END_TEXT, 'end.mp4', END_DUR, bg='1a0a00')
if os.path.exists(seg) and os.path.getsize(seg) > 0:
    segments.append(seg)

total_dur = sum(get_duration(s) for s in segments)
print(f'\n✓ {len(segments)} סגמנטים | {total_dur:.0f}ש ({total_dur/60:.2f} דקות)')

In [ ]:
# תא 8 — שרשור + מוזיקה = סרטון סופי
list_file = f'{WORK_DIR}/concat_list.txt'
with open(list_file, 'w') as f:
    for s in segments:
        f.write(f"file '{s}'\n")

silent = f'{WORK_DIR}/silent_video.mp4'
if os.path.exists(silent): os.remove(silent)
print('שרשור...')
run(
    f'ffmpeg -y -f concat -safe 0 -i "{list_file}" '
    f'-c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p "{silent}"',
    'concat'
)

# מוזיקה — באורך הוידאו המלא
actual_dur = get_duration(silent)
print(f'אורך וידאו: {actual_dur:.1f}ש')

music_out = f'{WORK_DIR}/mixed_audio.aac'
if os.path.exists(music_out): os.remove(music_out)
music = mix_music(actual_dur)

FINAL = f'{OUTPUT_DIR}/mazel_tov_final.mp4'
if os.path.exists(FINAL): os.remove(FINAL)
run(
    f'ffmpeg -y -i "{silent}" -i "{music}" '
    f'-c:v copy -c:a aac -b:a 192k -shortest "{FINAL}"',
    'final'
)

size = os.path.getsize(FINAL) / 1024 / 1024
dur  = get_duration(FINAL)
print(f'\n✓ הסרטון מוכן! {size:.0f}MB | {dur/60:.2f} דקות')

In [ ]:
# תא 9 — שמירה ל-Drive
import shutil
DRIVE_OUT = f'{DRIVE_PHOTOS}/mazel_tov_final.mp4'
print('שומר ל-Drive...')
shutil.copy2(FINAL, DRIVE_OUT)
print(f'✓ נשמר: {DRIVE_OUT}')
print(f'   גודל: {os.path.getsize(DRIVE_OUT)/1024/1024:.0f}MB')